# 05_goog_research — GOOG 圈内能力建设

> 配套 reflection: `reflections/goog/01-report-evaluation.md`

本 notebook 的作用：用数据补全报告**没说**的 8 个 missing topics。

**研究问题清单**（Q1–Q8）见 reflection 文档第三部分。

---

## Stage 1 — 估值数据（Q1, Q2, Q6, Q8）

本 stage 用 yfinance 拉取以下数据：
1. GOOG 当前估值（P/E, P/FCF, EV/EBITDA, FCF yield）
2. GOOG 5 年估值历史区间（确定当前在历史的什么分位）
3. 同业对比：MSFT / META / AMZN / AAPL
4. Capex / OCF 比例趋势（资本纪律检查）
5. 股份回购 / 分红 / FCF 转化

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

PEERS = ['GOOG', 'MSFT', 'META', 'AMZN', 'AAPL']
TODAY = datetime.now().strftime('%Y-%m-%d')
print(f'分析日期: {TODAY}')

### 1.1 当前估值快照（Q1, Q2）

拉取每只股票的当前快照：market cap, P/E (TTM/Forward), P/S, P/B, EV/EBITDA。

In [ ]:
def snapshot(ticker):
    t = yf.Ticker(ticker)
    info = t.info
    return {
        'ticker': ticker,
        'price': info.get('currentPrice'),
        'market_cap_B': info.get('marketCap', 0) / 1e9,
        'pe_ttm': info.get('trailingPE'),
        'pe_forward': info.get('forwardPE'),
        'ps': info.get('priceToSalesTrailing12Months'),
        'pb': info.get('priceToBook'),
        'ev_ebitda': info.get('enterpriseToEbitda'),
        'profit_margin': info.get('profitMargins'),
        'op_margin': info.get('operatingMargins'),
        'roe': info.get('returnOnEquity'),
        'fcf_B': info.get('freeCashflow', 0) / 1e9,
        'div_yield': info.get('dividendYield'),
    }

snap = pd.DataFrame([snapshot(t) for t in PEERS])
# FCF yield = FCF / Market Cap
snap['fcf_yield'] = snap['fcf_B'] / snap['market_cap_B']
snap

**判读：**
- GOOG 的 P/E TTM 在同业中排第几？
- GOOG 的 FCF yield vs 同业差距？(FCF yield 比同业低 200bps+ → Q2 证伪条件触发)
- GOOG 的 EV/EBITDA 在同业中排第几？

### 1.2 GOOG 自身 5 年估值历史（Q1）

用 P/E 历史构建当前在 5 年区间的分位数。

yfinance 不直接给历史 P/E，需要用 (price * shares) / earnings_ttm 自己算。
近似做法：用 P/E forward × 价格变动来回推历史 P/E 区间，或用 macrotrends.net 数据手工补。

In [ ]:
# 简化版：用价格 / 滚动 EPS（quarterly earnings 推算 TTM）
goog = yf.Ticker('GOOG')

# 价格历史
px = goog.history(period='5y', auto_adjust=True)['Close']

# 季度收益（quarterly_income_stmt 给最近 4 个季度，需要从其他来源补长历史）
# 这里只能给出近期估算
qis = goog.quarterly_income_stmt
print('季度财务（最近 4 个季度）：')
print(qis.loc[['Net Income', 'Operating Income', 'Total Revenue']] if not qis.empty else 'N/A')

print(f'\n当前价格: ${px.iloc[-1]:.2f}')
print(f'52W 高: ${px.tail(252).max():.2f}')
print(f'52W 低: ${px.tail(252).min():.2f}')
print(f'5Y 高: ${px.max():.2f}')
print(f'5Y 低: ${px.min():.2f}')

# 当前价格在 5Y 区间的分位数
pct = (px.iloc[-1] - px.min()) / (px.max() - px.min()) * 100
print(f'当前价在 5Y 价格区间的: {pct:.1f}% 分位')
print('注：价格分位 ≠ 估值分位（因为 EPS 也在变）')
print('真正的估值分位需要拉历史 EPS，建议用 macrotrends.net 或 stockanalysis.com 手工补充')

**手工补全任务：**
访问 https://www.macrotrends.net/stocks/charts/GOOG/alphabet/pe-ratio 查看：
- 5 年 P/E 区间的最高 / 最低 / 中位数
- 当前 P/E 在历史的什么位置

记录到 `reflections/goog/01-report-evaluation.md` 的 Q1 答案处。

### 1.3 Capex / OCF 趋势（Q6 — 资本纪律检查）

Capex 翻倍到 $180B+，但只有当 OCF 也跟得上时，资本纪律才不会塌。

**警戒线**：Capex / OCF > 80% → 资本纪律警戒

In [ ]:
# 现金流量表
cf = goog.cashflow  # 年度
if not cf.empty:
    print('GOOG 年度现金流（最近几年）：')
    rows = ['Operating Cash Flow', 'Capital Expenditure', 'Free Cash Flow', 'Repurchase Of Capital Stock']
    for row in rows:
        if row in cf.index:
            print(f'\n{row}:')
            print((cf.loc[row] / 1e9).map('{:.1f}B'.format))
    
    if 'Operating Cash Flow' in cf.index and 'Capital Expenditure' in cf.index:
        ratio = abs(cf.loc['Capital Expenditure']) / cf.loc['Operating Cash Flow']
        print('\nCapex / OCF 比例:')
        print(ratio.map('{:.1%}'.format))
        print('\n警戒线: > 80% 表示资本纪律压力')

### 1.4 同业 Capex 对比

In [ ]:
def capex_ratio(ticker):
    t = yf.Ticker(ticker)
    cf = t.cashflow
    if cf.empty:
        return None
    if 'Operating Cash Flow' in cf.index and 'Capital Expenditure' in cf.index:
        latest_year = cf.columns[0]
        ocf = cf.loc['Operating Cash Flow', latest_year]
        capex = abs(cf.loc['Capital Expenditure', latest_year])
        return {
            'ticker': ticker,
            'year': latest_year.year,
            'ocf_B': ocf / 1e9,
            'capex_B': capex / 1e9,
            'capex_pct_ocf': capex / ocf,
        }
    return None

results = [capex_ratio(t) for t in PEERS]
results = [r for r in results if r]
pd.DataFrame(results)

## Stage 2 — 待办（手工 / 网络补全）

yfinance 拿不到的数据，需要其他来源：

### Q3: DOJ 搜索案进展（反垄断尾部风险）
查询点：
- 案号：USA v. Google LLC (1:20-cv-03010)
- 2024 年判决要点：搜索分发被认定垄断
- 2025 年 remedy 阶段：Chrome 拆分 / 广告业务拆分提案
- 当前进展：?
- 资料来源：https://www.justice.gov/atr/case/us-and-plaintiff-states-v-google-llc-search

### Q4: 搜索广告 RPM
Q1 2026 财报会议是否披露：
- 搜索广告 cost-per-click 同比变化？
- 单位查询广告营收（RPM）？
- AI Overviews 已覆盖多少比例查询？覆盖部分 vs 未覆盖部分的 RPM 对比？
- 资料来源：Alphabet Q1 2026 10-Q + earnings call transcript

### Q7: Gemini 在 LMSys / Chatbot Arena 排名
- https://lmsys.org/leaderboard/
- 当前 Gemini 2.5 Pro 排名？相对 GPT-5 / Claude 3.5 Sonnet / Llama 4 / DeepSeek V4？

### Q8: Anthropic 合同条款
- 是否独占？(Anthropic 同时在 AWS Bedrock 上分发 Claude，本身就说明非独占)
- 21B / 100 万颗芯片到 2027 年 = 每颗均价 $21K？合理吗？

---

## Stage 3 — Bear Case 输入（待）

下次研究的输入材料：
1. 一份显式看空 GOOG 的报告（Wedbush / Needham 偶尔发布）
2. DOJ remedy 听证会文件
3. SemiAnalysis 关于 TPU vs NVIDIA 的独立技术分析（不是博客）

**Bear case 至少需要回答**：
- 如果 DOJ 强制拆分广告 / Chrome，GOOG 估值如何重估？
- 如果 AI Overviews 让 RPM 同比下降 5%，2026 全年 EPS 影响？
- 如果 Cloud 增速从 63% 降到 30%，2027 折旧释放叠加下，OCF 还够覆盖 $180B Capex 吗？

---

## Stage 4 — thesis.md 输出（7 月初）

最终产出：`portfolio/single-stocks/GOOG-thesis.md`

结构：
1. **持有论点**（4 个可证伪的 bull points）
2. **Invalidation 条件**（每个对应一个 bear scenario）
3. **决策**：保持 5% / 减为 2.5% + QQQ 2.5% / 完全替换 QQQ
4. **止损规则**：跌 -40% from peak / -60% from peak（继承 V5C 2.0 设计）

## 研究纪律提醒

1. **不在本 notebook 里回答 "GOOG 该不该持有 5%"** —— 那是 7/29 thesis.md 的事。
2. **不基于本 notebook 的中间结果改 V5C 3.0** —— 5% 已定。
3. **每个 cell 的输出都要写一句话结论**，记录在 `reflections/goog/` 下对应的 NN-NNN.md。
4. **Bear case 优先级 > Bull case** —— 已经有一份多头报告了，本 notebook 主要是反向找漏洞。